# Thresholding Matrices

This is an analysis of the count of network motifs with different thresholding parameters.

In [ ]:
# Required packages

display("text/html", "<style>.container { width:100% !important; }</style>")

# Julia dependencies for plotting, stats, and experiments
using Plots, LaTeXStrings, Distributions, DoubleFloats, Plots.PlotMeasures, LinearAlgebra,
    DataFrames, CSV, LsqFit, StatsBase, Distances, Parameters, Random, StatsBase, Graphs, 
    GraphPlot, Base.Threads, Printf, JLD2

const ROOT_DIR = normpath(joinpath(pwd(), "..", ".."))
const SRC_DIR = joinpath(ROOT_DIR, "src")
const SCRIPTS_DIR = joinpath(ROOT_DIR, "scripts")
const SCRATCH_DIR = mktempdir()

# Local modules
include(joinpath(SRC_DIR, "wagner_algorithm.jl"))
using ..BooleanNetwork: SimulationParameters, SimulationData
using ..BooleanNetwork: run_simulation
include(joinpath(SRC_DIR, "data_processing.jl"))
using ..CustomStats: summarize_history, summarize_simulation_run, alignment_score, compute_mut_robustness
include(joinpath(SCRIPTS_DIR, "run_experiments.jl"))
using ..ExperimentHandler: Experiment1Config, Experiment2Config, Experiment1Result, Experiment2Result, MotifStats
using ..ExperimentHandler: run_experiment_1, run_experiment_2, build_noise_scenarios
using ..ExperimentHandler: compute_ffl_type_counts, compute_fbck_type_counts

# Utilities
using ProgressMeter
using Base.Threads: Atomic, atomic_add!

# Python bridge for motif search utilities
using PyCall
py"""
import sys
import os
sys.path.append(os.path.abspath("../src/"))
from motif_search import *
"""


default(dpi=900, legendfontsize=24, guidefont=font(24), tickfont=font(24), margin=8Plots.mm, grid=false)

"""
    save_plot(plot_obj, filename)

Save a figure to the figures directory and print the output path.
"""
function save_plot(plot_obj, filename)
    filepath = joinpath(FIGURES_DIR, filename)
    savefig(plot_obj, filepath)
    println("Saved $(filepath)")
end

In [ ]:
# Load data
key_1_name = "exp_1_standard"
exp_1_result = load(joinpath(ROOT_DIR, "notebooks", "data", "experiments_results.jld2"),key_1_name)
exp_1_matrices = exp_1_result.matrix_samples;
size(exp_1_matrices)  # (n_scenarios, trials, sample_points, sample_sizes).

## Noiseless

In [ ]:
original_matrices = exp_1_matrices[1, :,end, :];  # (trials, sample size)
results_counts = compute_ffl_type_counts(original_matrices[1, :], nothing);

In [ ]:
function scatterplot_ffls(matrix_vec, title = "")
    sorted_keys = sort(collect(keys(matrix_vec)))
    sorted_values = [matrix_vec[k] for k in sorted_keys]
    averages = mean.(sorted_values)
    stds = std.(sorted_values, corrected=true)

    Plots.theme(:dao)
    p1 = scatter(sorted_keys, averages,
        yerror=stds,
        legend=false,
        markersize=8,
        grid=false,
        minorgrid=false,
        markerstrokewidth=1.0,
        markershape=:x,
        errorbarlinewidth=0.5,
        errorbarcapsize=4,
        dpi=600,
        xrotation=90,
        title=title)
    return p1
end

function threshold_mat(matrix, tol)
    thresholded = copy(matrix)
    thresholded[abs.(matrix) .<= tol] .= 0
    return thresholded
end

original_matrices = exp_1_matrices[1, :, end, :]  # (trials, sample size)

for tol in [0.0, 0.25, 0.5, 1.0,1.5]
    threshold_1 = threshold_mat.(original_matrices[1, :], tol)
    results_counts = compute_ffl_type_counts(threshold_1, nothing)

    p1 = scatterplot_ffls(results_counts[1], "Raw Counts")
    p2 = scatterplot_ffls(results_counts[2], "Relative Frequencies")
    p12 = plot!(p1, p2, layout=(2,1), size=(800,1200), plot_title = "Threshold = $(round(tol,digits=3))")
    savefig(p12,"thresholding/noiseless_relative_$tol.png")
    display(p12)
end

## Noisy

In [ ]:
original_matrices = exp_1_matrices[4, :, end, :];  # (trials, sample size)

In [ ]:
original_matrices = exp_1_matrices[4, :, end, :]  # (trials, sample size)

for tol in [0.0, 0.25, 0.5, 1.0, 1.5]
    threshold_1 = threshold_mat.(original_matrices[1, :], tol)
    results_counts = compute_ffl_type_counts(threshold_1, nothing)

    p1 = scatterplot_ffls(results_counts[1], "Raw Counts")
    p2 = scatterplot_ffls(results_counts[2], "Relative Frequencies")
    p12 = plot!(p1, p2, layout=(2, 1), size=(800, 1200), plot_title="Threshold = $(round(tol,digits=3))")
    savefig(p12, "thresholding/noisy_relative_$tol.png")
    display(p12)
end

## Very Noisy

In [ ]:
original_matrices = exp_1_matrices[7, :, end, :]  # (trials, sample size)

for tol in [0.0, 0.25, 0.5, 1.0, 1.5]
    threshold_1 = threshold_mat.(original_matrices[1, :], tol)
    results_counts = compute_ffl_type_counts(threshold_1, nothing)

    p1 = scatterplot_ffls(results_counts[1], "Raw Counts")
    p2 = scatterplot_ffls(results_counts[2], "Relative Frequencies")
    p12 = plot!(p1, p2, layout=(2, 1), size=(800, 1200), plot_title=L"\theta = 8.0." * " Threshold = $(round(tol,digits=3))")
    savefig(p12, "thresholding/very_noisy_relative_$tol.png")

    display(p12)
end